# cjm-fasthtml-workflow-transcript-decomp

> A FastHTML-based human-in-the-loop workflow for decomposing raw transcripts into traversable context graph spines with precise text segmentation and audio alignment.

## Install


```bash
pip install cjm_fasthtml_workflow_transcript_decomp
```

## Project Structure

```
nbs/
├── combined/ (5)
│   ├── handlers.ipynb         # Handler wrappers for cross-domain coordination (alignment status updates)
│   ├── helpers.ipynb          # State extraction helpers for cross-domain coordination in Phase 2 combined step
│   ├── html_ids.ipynb         # HTML ID constants for Phase 2 Shell: Dual-Column Layout shared chrome
│   ├── keyboard_config.ipynb  # Shared keyboard navigation configuration for the combined Phase 2 step
│   └── step_combined.ipynb    # Phase 2 combined step renderer: dual-column layout for Segment & Align
├── core/ (1)
│   └── config.ipynb  # Configuration dataclass for structure decomposition workflow
├── routes/ (6)
│   ├── core/ (5)
│   │   ├── audio.ipynb    # Audio file serving route handlers for alignment playback
│   │   ├── chrome.ipynb   # Shared chrome switching route handlers for the combined Phase 2 step
│   │   ├── init.ipynb     # Router assembly for core workflow routes
│   │   ├── sources.ipynb  # Source data retrieval route handlers
│   │   └── status.ipynb   # Workflow status and lifecycle route handlers
│   └── init.ipynb  # Router initialization for the structure decomposition workflow
└── workflow/ (1)
    └── workflow.ipynb  # Main workflow class for structure decomposition
```

Total: 13 notebooks across 4 directories

## Module Dependencies

```mermaid
graph LR
    combined_handlers[combined.handlers<br/>handlers]
    combined_helpers[combined.helpers<br/>helpers]
    combined_html_ids[combined.html_ids<br/>html_ids]
    combined_keyboard_config[combined.keyboard_config<br/>keyboard_config]
    combined_step_combined[combined.step_combined<br/>step_combined]
    core_config[core.config<br/>config]
    routes_core_audio[routes.core.audio<br/>audio]
    routes_core_chrome[routes.core.chrome<br/>chrome]
    routes_core_init[routes.core.init<br/>init]
    routes_core_sources[routes.core.sources<br/>sources]
    routes_core_status[routes.core.status<br/>status]
    routes_init[routes.init<br/>init]
    workflow_workflow[workflow.workflow<br/>workflow]

    combined_handlers --> combined_step_combined
    combined_handlers --> combined_html_ids
    combined_handlers --> combined_keyboard_config
    combined_keyboard_config --> combined_html_ids
    combined_step_combined --> combined_helpers
    combined_step_combined --> combined_html_ids
    combined_step_combined --> combined_keyboard_config
    routes_core_audio --> workflow_workflow
    routes_core_chrome --> combined_step_combined
    routes_core_chrome --> combined_html_ids
    routes_core_chrome --> combined_keyboard_config
    routes_core_chrome --> workflow_workflow
    routes_core_init --> workflow_workflow
    routes_core_init --> routes_core_chrome
    routes_core_init --> routes_core_sources
    routes_core_init --> routes_core_audio
    routes_core_init --> routes_core_status
    routes_core_sources --> workflow_workflow
    routes_core_status --> workflow_workflow
    routes_init --> combined_handlers
    routes_init --> workflow_workflow
    routes_init --> routes_core_init
    workflow_workflow --> core_config
    workflow_workflow --> combined_step_combined
```

*24 cross-module dependencies detected*

## CLI Reference

No CLI commands found in this project.

## Module Overview

Detailed documentation for each module in the project:

### audio (`audio.ipynb`)
> Audio file serving route handlers for alignment playback

#### Import

```python
from cjm_fasthtml_workflow_transcript_decomp.routes.core.audio import (
    DEBUG_AUDIO,
    init_audio_router
)
```

#### Functions

```python
def _handle_audio_src(
    workflow:StructureDecompWorkflow,  # The workflow instance
    sess,  # FastHTML session object
    path:str=None,  # Audio file path (from query parameter)
):  # Audio file response or 404
    "Serve an audio file by path for Web Audio API playback."
```

```python
def init_audio_router(
    workflow: StructureDecompWorkflow,  # The workflow instance
    prefix: str,  # Route prefix (e.g., "/workflow/core/audio")
) -> Tuple[APIRouter, Dict[str, Callable]]:  # (router, route_dict)
    "Initialize audio serving routes."
```

#### Variables

```python
DEBUG_AUDIO = False
```

### chrome (`chrome.ipynb`)
> Shared chrome switching route handlers for the combined Phase 2 step

#### Import

```python
from cjm_fasthtml_workflow_transcript_decomp.routes.core.chrome import (
    DEBUG_SWITCH_CHROME,
    init_chrome_router
)
```

#### Functions

```python
async def _handle_switch_chrome(
    workflow:StructureDecompWorkflow,  # The workflow instance
    request,  # FastHTML request object
    sess,  # FastHTML session object
    seg_urls:SegmentationUrls,  # URL bundle for segmentation routes
    align_urls:AlignmentUrls,  # URL bundle for alignment routes
) -> tuple:  # OOB swaps for shared chrome containers
    "Switch shared chrome content based on active column."
```

```python
def init_chrome_router(
    workflow: StructureDecompWorkflow,  # The workflow instance
    prefix: str,  # Route prefix (e.g., "/workflow/core/chrome")
) -> Tuple[APIRouter, Dict[str, Callable]]:  # (router, route_dict)
    "Initialize chrome switching routes."
```

#### Variables

```python
DEBUG_SWITCH_CHROME = False
```

### config (`config.ipynb`)
> Configuration dataclass for structure decomposition workflow

#### Import

```python
from cjm_fasthtml_workflow_transcript_decomp.core.config import (
    DEFAULT_WORKFLOW_CONFIG_DIR,
    StructureDecompWorkflowConfig
)
```
#### Classes

```python
@dataclass
class StructureDecompWorkflowConfig:
    "Configuration for structure decomposition workflow."
    
    workflow_id: str = 'structure_decomposition'  # Unique identifier for this workflow
    route_prefix: str = '/structure_decomp'  # Base URL prefix for workflow routes
    stepflow_prefix: str = '/flow'  # Sub-prefix for StepFlow routes
    container_id: str = 'sd-workflow-container'  # HTML ID for main workflow container
    show_progress: bool = True  # Show step progress indicator in StepFlow
    max_history_depth: int = 500  # Maximum undo history entries for decomposition phase
    text_plugin: str = 'cjm-text-plugin-nltk'  # Text processing plugin for sentence splitting
    vad_plugin: str = 'cjm-media-plugin-silero-vad'  # VAD plugin for audio alignment
    graph_plugin: str = 'cjm-graph-plugin-sqlite'  # Graph plugin for storage
    source_categories: List[str] = field(...)  # Categories for source plugins
    no_plugins_redirect: Optional[str]  # URL to redirect when required plugins unavailable
    state_db_path: Optional[Path]  # Path to SQLite state database (uses default if None)
    config_dir: Path = field(...)  # Directory for workflow settings
    
    def get_full_stepflow_prefix(self) -> str:  # Combined route_prefix + stepflow_prefix
            """Get the full prefix for the StepFlow router."""
            return f"{self.route_prefix}{self.stepflow_prefix}"
        
        def get_state_db_path(self) -> Path:  # Resolved path to state database
        "Get the full prefix for the StepFlow router."
    
    def get_state_db_path(self) -> Path:  # Resolved path to state database
            """Get the path to the SQLite state database."""
            if self.state_db_path
        "Get the path to the SQLite state database."
```

#### Variables

```python
DEFAULT_WORKFLOW_CONFIG_DIR
```

### handlers (`handlers.ipynb`)
> Handler wrappers for cross-domain coordination (alignment status updates)

#### Import

```python
from cjm_fasthtml_workflow_transcript_decomp.combined.handlers import (
    wrapped_seg_split,
    wrapped_seg_merge,
    wrapped_seg_undo,
    wrapped_seg_reset,
    wrapped_seg_ai_split,
    wrap_seg_mutation_handler,
    wrap_align_mutation_handler,
    create_seg_init_chrome_wrapper,
    create_align_init_chrome_wrapper
)
```

#### Functions

```python
def _find_session_id(args, kwargs):
    """Find session_id from args or kwargs."""
    # First check kwargs
    if 'sess' in kwargs
    "Find session_id from args or kwargs."
```

```python
def wrap_seg_mutation_handler(
    handler: Callable,  # Handler function to wrap
) -> Callable:  # Wrapped handler that appends alignment status OOB
    """
    Wrap a segmentation mutation handler to add alignment status OOB.
    
    The handler is expected to take (state_store, workflow_id, ...) as first params.
    """
```

```python
def wrap_align_mutation_handler(
    handler: Callable,  # Handler function to wrap
) -> Callable:  # Wrapped handler that appends alignment status OOB
    """
    Wrap an alignment mutation handler to add alignment status OOB.
    
    The handler is expected to take (state_store, workflow_id, ...) as first params.
    """
```

```python
def create_seg_init_chrome_wrapper(
    align_urls:AlignmentUrls,  # URL bundle for alignment routes (for KB system)
    switch_chrome_url:str,  # URL for chrome switching (for KB system)
) -> Callable:  # Wrapped handler that builds KB system and shared chrome
    """
    Create a wrapper for seg init that builds combined KB system and shared chrome.
    
    This is a factory that captures the URLs needed for KB system assembly.
    """
```

```python
def create_align_init_chrome_wrapper() -> Callable:  # Wrapped handler that adds alignment status
    """Create a wrapper for align init that adds mini-stats and alignment status.
    
    Alignment init is simpler than seg init - it doesn't need to build the
    full KB system (seg init handles that). It just updates alignment-specific
    chrome and the alignment status badge.
    """
    async def wrapped_align_init(
        state_store:WorkflowStateStore,
        workflow_id:str,
        source_service:SourceService,
        alignment_service:AlignmentService,
        request:Any,
        sess:Any,
        urls:AlignmentUrls,
        visible_count:int=5,
        card_width:int=40,
    )
    """
    Create a wrapper for align init that adds mini-stats and alignment status.
    
    Alignment init is simpler than seg init - it doesn't need to build the
    full KB system (seg init handles that). It just updates alignment-specific
    chrome and the alignment status badge.
    """
```


### helpers (`helpers.ipynb`)
> State extraction helpers for cross-domain coordination in Phase 2 combined step

#### Import

```python
from cjm_fasthtml_workflow_transcript_decomp.combined.helpers import (
    SEG_DEFAULT_VISIBLE_COUNT,
    SEG_DEFAULT_CARD_WIDTH,
    ALIGN_DEFAULT_VISIBLE_COUNT,
    ALIGN_DEFAULT_CARD_WIDTH,
    check_alignment_ready,
    extract_seg_state,
    extract_alignment_state,
    get_segment_count,
    get_chunk_count
)
```

#### Functions

```python
def check_alignment_ready(
    segment_count:int,  # Number of text segments
    chunk_count:int,  # Number of VAD chunks
) -> bool:  # True if counts match for 1:1 alignment
    "Check if segment and VAD chunk counts match for 1:1 alignment."
```

```python
def extract_seg_state(
    ctx:InteractionContext,  # Interaction context with state
) -> Dict[str, Any]:  # Extracted state values
    "Extract segmentation state as explicit values for renderers."
```

```python
def extract_alignment_state(
    ctx:InteractionContext,  # Interaction context with state
) -> Dict[str, Any]:  # Extracted state values
    "Extract alignment state as explicit values for renderers."
```

```python
def get_segment_count(
    ctx:InteractionContext,  # Interaction context with state
) -> int:  # Number of segments
    "Get segment count from state without full extraction."
```

```python
def get_chunk_count(
    ctx:InteractionContext,  # Interaction context with state
) -> int:  # Number of VAD chunks
    "Get VAD chunk count from state without full extraction."
```

#### Variables

```python
SEG_DEFAULT_VISIBLE_COUNT = 3
SEG_DEFAULT_CARD_WIDTH = 80
ALIGN_DEFAULT_VISIBLE_COUNT = 5
ALIGN_DEFAULT_CARD_WIDTH = 40
```

### html_ids (`html_ids.ipynb`)
> HTML ID constants for Phase 2 Shell: Dual-Column Layout shared chrome

#### Import

```python
from cjm_fasthtml_workflow_transcript_decomp.combined.html_ids import (
    CombinedHtmlIds
)
```
#### Classes

```python
class CombinedHtmlIds:
    "HTML ID constants for Phase 2 Shell: Dual-Column Layout shared chrome."
    
    def as_selector(
            id_str:str  # The HTML ID to convert
        ) -> str:  # CSS selector with # prefix
        "Convert an ID to a CSS selector format."
```


### init (`init.ipynb`)
> Router assembly for core workflow routes

#### Import

```python
from cjm_fasthtml_workflow_transcript_decomp.routes.core.init import (
    init_core_routers
)
```

#### Functions

```python
def init_core_routers(
    workflow: StructureDecompWorkflow,  # The workflow instance
    prefix: str,  # Base prefix for core routes (e.g., "/workflow/core")
) -> Tuple[List[APIRouter], Dict[str, Callable]]:  # (routers, merged_routes)
    "Initialize and return all core workflow routers."
```


### init (`init.ipynb`)
> Router initialization for the structure decomposition workflow

#### Import

```python
from cjm_fasthtml_workflow_transcript_decomp.routes.init import (
    init_routers
)
```

#### Functions

```python
def init_routers(
    workflow: "StructureDecompWorkflow",  # The workflow instance
) -> List[APIRouter]:  # List of configured routers
    "Initialize and return all workflow routers."
```


### keyboard_config (`keyboard_config.ipynb`)
> Shared keyboard navigation configuration for the combined Phase 2 step

#### Import

```python
from cjm_fasthtml_workflow_transcript_decomp.combined.keyboard_config import (
    DEBUG_KB_SYSTEM,
    ZONE_CHANGE_CALLBACK,
    SWITCH_CHROME_BTN_ID,
    render_keyboard_hints_collapsible,
    build_combined_kb_system,
    generate_zone_change_js
)
```

#### Functions

```python
def render_keyboard_hints_collapsible(
    manager:ZoneManager,  # Keyboard zone manager with actions configured
    container_id:str="sd-keyboard-hints",  # HTML ID for the hints container
    include_zone_switch:bool=False,  # Whether to include zone switch hints
) -> Any:  # Collapsible keyboard hints component
    "Render keyboard shortcut hints in a collapsible DaisyUI collapse."
```

```python
def build_combined_kb_system(
    seg_urls:SegmentationUrls,  # URL bundle for segmentation routes
    align_urls:AlignmentUrls,  # URL bundle for alignment routes
) -> Tuple[ZoneManager, Any]:  # (keyboard manager, rendered keyboard system)
    "Build combined keyboard system with segmentation and alignment zones."
```

```python
def generate_zone_change_js(
    switch_chrome_url:str="",  # URL for chrome swap handler (empty = no swap)
) -> Script:  # Script element with zone change callback and click handlers
    "Generate JavaScript for zone change handling and column click handlers."
```

#### Variables

```python
DEBUG_KB_SYSTEM = True
ZONE_CHANGE_CALLBACK = 'onCombinedZoneChange'
SWITCH_CHROME_BTN_ID = 'sd-switch-chrome-btn'
```

### sources (`sources.ipynb`)
> Source data retrieval route handlers

#### Import

```python
from cjm_fasthtml_workflow_transcript_decomp.routes.core.sources import (
    init_sources_router
)
```

#### Functions

```python
def _handle_get_sources(
    workflow: StructureDecompWorkflow,  # The workflow instance
    request,  # FastHTML request object
    provider_id: str = None,  # Optional plugin name filter
    limit: int = 50  # Maximum number of results
):  # JSON response with transcription sources
    "Get available transcription sources."
```

```python
def init_sources_router(
    workflow: StructureDecompWorkflow,  # The workflow instance
    prefix: str,  # Route prefix (e.g., "/workflow/core/sources")
) -> Tuple[APIRouter, Dict[str, Callable]]:  # (router, route_dict)
    "Initialize source data routes."
```


### status (`status.ipynb`)
> Workflow status and lifecycle route handlers

#### Import

```python
from cjm_fasthtml_workflow_transcript_decomp.routes.core.status import (
    init_status_router
)
```

#### Functions

```python
async def _handle_current_status(
    workflow: StructureDecompWorkflow,  # The workflow instance
    request,  # FastHTML request object
    sess  # FastHTML session object
):  # Appropriate UI component based on current state
    "Return current workflow status - determines what to show."
```

```python
async def _handle_reset(
    workflow: StructureDecompWorkflow,  # The workflow instance
    request,  # FastHTML request object
    sess  # FastHTML session object
):  # StepFlow start view
    "Reset workflow and return to start."
```

```python
def init_status_router(
    workflow: StructureDecompWorkflow,  # The workflow instance
    prefix: str,  # Route prefix (e.g., "/workflow/core/status")
) -> Tuple[APIRouter, Dict[str, Callable]]:  # (router, route_dict)
    "Initialize status and lifecycle routes."
```


### step_combined (`step_combined.ipynb`)
> Phase 2 combined step renderer: dual-column layout for Segment & Align

#### Import

```python
from cjm_fasthtml_workflow_transcript_decomp.combined.step_combined import (
    DEBUG_COMBINED_RENDER,
    render_seg_mini_stats_badge,
    render_align_mini_stats_badge,
    render_alignment_status_text,
    render_alignment_status,
    render_footer_inner_content,
    render_combined_step
)
```

#### Functions

```python
def _render_column_header(
    title:str,  # Column title (e.g., "Text Decomposition")
    stats_id:str,  # HTML ID for the mini-stats badge area
    header_id:str,  # HTML ID for the column header container
    initial_text:str="--",  # Initial text for the mini-stats badge
) -> Any:  # Column header component
    "Render a column header with title and mini-stats badge."
```

```python
def render_seg_mini_stats_badge(
    segments:List[TextSegment],  # Current segments
    oob:bool=False,  # Whether to render as OOB swap
) -> Any:  # Mini-stats badge Span
    "Render the segmentation mini-stats badge for the column header."
```

```python
def render_align_mini_stats_badge(
    chunks:List[VADChunk],  # Current VAD chunks
    oob:bool=False,  # Whether to render as OOB swap
) -> Any:  # Mini-stats badge Span
    "Render the alignment mini-stats badge for the column header."
```

```python
def render_alignment_status_text(
    segment_count:int,  # Number of text segments
    chunk_count:int,  # Number of VAD chunks
) -> str:  # Status message text
    "Generate alignment status message based on segment and VAD chunk counts."
```

```python
def render_alignment_status(
    segment_count:int,  # Number of text segments
    chunk_count:int,  # Number of VAD chunks
    oob:bool=False,  # Whether to render as OOB swap
) -> Any:  # Alignment status badge component
    "Render the alignment status indicator badge."
```

```python
def render_footer_inner_content(
    column_footer:Any,  # Column-specific footer content (decomp or align)
    segment_count:int,  # Number of text segments
    chunk_count:int,  # Number of VAD chunks
) -> Any:  # Styled wrapper div with column footer and alignment status
    """
    Render the footer inner content with consistent styling.
    
    This ensures the footer layout (justify-between) is preserved across
    all OOB swaps. Both the column-specific footer content and the
    alignment status indicator are wrapped in a flex container.
    """
```

```python
def _placeholder(
    text:str,  # Placeholder message
) -> Any:  # Styled placeholder paragraph
    "Render a placeholder text element for uninitialized chrome containers."
```

```python
def _render_shared_chrome(
    seg_state:dict=None,  # Extracted segmentation state (None = show placeholders)
    align_state:dict=None,  # Extracted alignment state (None = no VAD data yet)
    urls:SegmentationUrls=None,  # Segmentation URL bundle (required when seg_state provided)
    kb_manager:Any=None,  # Keyboard manager (required when seg_state provided)
) -> tuple:  # (hints, toolbar, controls, footer)
    """
    Render shared chrome containers, populated with segmentation content when initialized.
    
    Takes extracted state dicts from `extract_seg_state()` and `extract_alignment_state()`
    which contain deserialized TextSegment and VADChunk objects.
    """
```

```python
def _render_seg_column(
    is_active:bool=True,  # Whether this column is initially active
    column_body:Any=None,  # Pre-rendered column body (None = not initialized)
    mini_stats_text:str="--",  # Mini-stats badge text
    init_url:str="",  # URL for auto-trigger initialization
) -> Any:  # Left column component
    "Render the left segmentation column."
```

```python
def _render_alignment_column(
    is_active:bool=False,  # Whether this column is initially active
    column_body:Any=None,  # Pre-rendered column body (None = not initialized)
    mini_stats_text:str="--",  # Mini-stats badge text
    init_url:str="",  # URL for auto-trigger initialization
) -> Any:  # Right column component
    "Render the right alignment column."
```

```python
def _render_keyboard_system_container(
    kb_system:Any=None,  # Rendered keyboard system (None = empty container)
    oob:bool=False,  # Whether to render as OOB swap
) -> Any:  # Div with id=KEYBOARD_SYSTEM containing KB elements
    "Render stable container for keyboard navigation system elements."
```

```python
def render_combined_step(
    ctx:InteractionContext,  # Interaction context with state and data
    seg_urls:SegmentationUrls=None,  # URL bundle for segmentation routes
    align_urls:AlignmentUrls=None,  # URL bundle for alignment routes
    switch_chrome_url:str="",  # URL for chrome switching route
) -> Any:  # FastHTML component with full dual-column layout
    "Render Phase 2: Combined Segment & Align step with dual-column layout."
```

#### Variables

```python
DEBUG_COMBINED_RENDER = True
_FOOTER_INNER_CLS
_SEG_COLUMN_CLS
_ALIGNMENT_COLUMN_CLS
```

### workflow (`workflow.ipynb`)
> Main workflow class for structure decomposition

#### Import

```python
from cjm_fasthtml_workflow_transcript_decomp.workflow.workflow import (
    StructureDecompWorkflow
)
```

#### Functions

```python
@patch
def setup(
    self: StructureDecompWorkflow,
    app  # FastHTML application instance
) -> None
    "Initialize workflow with FastHTML app."
```

```python
@patch
def cleanup(
    self: StructureDecompWorkflow
) -> None
    "Clean up workflow resources."
```

```python
@patch
def get_routers(
    self: StructureDecompWorkflow
) -> List[APIRouter]:  # List of routers to register
    "Return all routers for registration with the app."
```

```python
@patch
def render_entry_point(
    self: StructureDecompWorkflow,
    request,  # FastHTML request object
    sess  # FastHTML session object
):  # FastHTML component
    "Render the workflow entry point for embedding."
```

```python
def _create_data_loaders(
    workflow: StructureDecompWorkflow  # Workflow instance for service access
):  # (load_sources, load_empty) callables
    "Create data loader functions for StepFlow steps."
```

```python
def _validate_always_true(
    state: Dict[str, Any]  # Workflow state dictionary
) -> bool:  # Always True
    "No-op validator for steps that are always valid."
```

```python
def _validate_selection(
    state: Dict[str, Any]  # Workflow state dictionary
) -> bool:  # True if at least one source is selected
    "Validate that sources have been selected."
```

```python
def _create_selection_renderer(
    workflow: StructureDecompWorkflow  # Workflow instance for URL access
):  # Render callable for StepFlow
    "Create render function for the source selection step."
```

```python
def _validate_segmentation(
    state: Dict[str, Any]  # Workflow state dictionary
) -> bool:  # True if segments and VAD chunks are 1:1 aligned
    "Validate that segmentation and alignment are complete."
```

```python
def _create_combined_renderer(
    workflow: StructureDecompWorkflow  # Workflow instance for URL access
):  # Render callable for StepFlow
    "Create render function for the segmentation & alignment step."
```

```python
def _create_review_renderer(
    workflow: StructureDecompWorkflow  # Workflow instance for URL and state access
):  # Render callable for StepFlow
    "Create render function for the review step."
```

```python
def _create_review_hook(
    workflow: StructureDecompWorkflow  # Workflow instance for service and state access
):  # on_leave callable
    "Create on_leave hook for the review step (commits to graph)."
```

```python
def _create_verify_renderer(
    workflow: StructureDecompWorkflow  # Workflow instance for URL access
):  # Render callable for StepFlow
    "Create render function for the verify step."
```

```python
def _create_verify_hooks(
    workflow: StructureDecompWorkflow  # Workflow instance for service and state access
):  # (on_enter_verify, on_complete) callables
    "Create lifecycle hooks for the verify step."
```

```python
@patch
def _create_step_flow(
    self: StructureDecompWorkflow
) -> StepFlow:  # Configured StepFlow instance
    "Create and configure the StepFlow instance."
```

```python
@patch
def _create_routers(
    self: StructureDecompWorkflow
) -> List[APIRouter]:  # List of configured APIRouters
    "Create the workflow's API routers."
```

#### Classes

```python
class _SessionStateStoreAdapter:
    def __init__(
        self,
        store: SQLiteWorkflowStateStore  # The pure, framework-agnostic store
    )
    "Adapter bridging StepFlow's sess-based protocol to the pure session_id-based store."
    
    def __init__(
            self,
            store: SQLiteWorkflowStateStore  # The pure, framework-agnostic store
        )
        "Wrap a pure store with session resolution."
    
    def get_current_step(
            self,
            flow_id: str,  # Workflow identifier
            sess: Any  # FastHTML session object
        ) -> Optional[str]:  # Current step ID or None
        "Get current step ID, resolving session from sess."
    
    def set_current_step(
            self,
            flow_id: str,  # Workflow identifier
            sess: Any,  # FastHTML session object
            step_id: str  # Step ID to set as current
        ) -> None
        "Set current step ID, resolving session from sess."
    
    def get_state(
            self,
            flow_id: str,  # Workflow identifier
            sess: Any  # FastHTML session object
        ) -> Dict[str, Any]:  # Workflow state dictionary
        "Get all workflow state, resolving session from sess."
    
    def update_state(
            self,
            flow_id: str,  # Workflow identifier
            sess: Any,  # FastHTML session object
            updates: Dict[str, Any]  # State updates to apply
        ) -> None
        "Update workflow state, resolving session from sess."
    
    def clear_state(
            self,
            flow_id: str,  # Workflow identifier
            sess: Any  # FastHTML session object
        ) -> None
        "Clear all workflow state, resolving session from sess."
```

```python
class StructureDecompWorkflow:
    def __init__(
        self,
        plugin_manager: PluginManager,  # Plugin manager from host application
        config: Optional[StructureDecompWorkflowConfig] = None  # Workflow configuration
    )
    "Self-contained structure decomposition workflow."
    
    def __init__(
            self,
            plugin_manager: PluginManager,  # Plugin manager from host application
            config: Optional[StructureDecompWorkflowConfig] = None  # Workflow configuration
        )
        "Initialize the workflow with injected PluginManager."
    
    def create_and_setup(
            cls,
            app,  # FastHTML application instance
            plugin_manager: PluginManager,  # Plugin manager from host application
            config: Optional[StructureDecompWorkflowConfig] = None  # Workflow configuration
        ) -> "StructureDecompWorkflow":  # Configured and setup workflow instance
        "Create, configure, and setup a workflow in one call."
    
    def plugin_manager(self) -> PluginManager:  # Plugin manager instance
            """Access to plugin manager."""
            return self._plugin_manager
        
        @property
        def source_service(self) -> SourceService:  # Source service instance
        "Access to plugin manager."
    
    def source_service(self) -> SourceService:  # Source service instance
            """Access to source service."""
            return self._source_service
        
        @property
        def segmentation_service(self) -> SegmentationService:  # Segmentation service instance
        "Access to source service."
    
    def segmentation_service(self) -> SegmentationService:  # Segmentation service instance
            """Access to segmentation service."""
            return self._segmentation_service
        
        @property
        def alignment_service(self) -> AlignmentService:  # Alignment service instance
        "Access to segmentation service."
    
    def alignment_service(self) -> AlignmentService:  # Alignment service instance
            """Access to alignment service."""
            return self._alignment_service
        
        @property
        def graph_service(self) -> GraphService:  # Graph service instance
        "Access to alignment service."
    
    def graph_service(self) -> GraphService:  # Graph service instance
            """Access to graph service."""
            return self._graph_service
        
        @property
        def verify_service(self) -> VerifyService:  # Verify service instance
        "Access to graph service."
    
    def verify_service(self) -> VerifyService:  # Verify service instance
            """Access to verify service."""
            return self._verify_service
        
        @property
        def state_store(self) -> SQLiteWorkflowStateStore:  # State store instance
        "Access to verify service."
    
    def state_store(self) -> SQLiteWorkflowStateStore:  # State store instance
            """Access to state store."""
            return self._state_store
        
        @property
        def routers(self) -> List[APIRouter]:  # List of workflow routers
        "Access to state store."
    
    def routers(self) -> List[APIRouter]:  # List of workflow routers
            """Access to workflow routers (excludes stepflow router)."""
            return self._routers
        
        @property
        def stepflow_router(self) -> APIRouter:  # StepFlow-generated router
        "Access to workflow routers (excludes stepflow router)."
    
    def stepflow_router(self) -> APIRouter:  # StepFlow-generated router
        "StepFlow-generated router."
```
